In [6]:
import pandas as pd

# TACHE 1 : CHARGEMENT, NETTOYAGE ET QUESTION 9
# 1. Charger les fichiers d'origine
reservations = pd.read_csv('./data/Valmont_reservations.csv')
hotels = pd.read_csv('./data/Valmont_hotels.csv')

print("Nombre de réservations :", len(reservations))
print("Nombre d'hôtels :", len(hotels))

# 2. Fusionner les deux tables
df = pd.merge(reservations, hotels, on='hotel_id', how='left')
print("Les deux fichiers ont été fusionnés.")
print("Nombre de lignes après la fusion :", len(df))

# 3. Supprimer les doublons
print("Nombre de doublons trouvés :", df.duplicated(subset=['reservation_id']).sum())
df = df.drop_duplicates(subset=['reservation_id'])
print("Nombre de lignes restantes :", len(df))

# 4. Gérer les valeurs manquantes
print("Valeurs manquantes avant le remplacement :")
print(df[['nb_personnes', 'petit_dejeuner']].isna().sum())

df['nb_personnes'] = df['nb_personnes'].fillna(df['nb_personnes'].median())
df['petit_dejeuner'] = df['petit_dejeuner'].fillna(df['petit_dejeuner'].mode()[0])
print(df[['nb_personnes', 'petit_dejeuner']].isna().sum())

# 5. Supprimer les valeurs aberrantes
df_clean = df[(df['prix_nuit_eur'] > 0) & (df['nb_nuits'] <= 30) & (df['nb_personnes'] > 0)].copy()
print("Nombre de réservations écartées :", len(df) - len(df_clean))
print("Nombre de réservations gardées :", len(df_clean))

# 6. Q9 (les réservations anormales à bloquer)
dossiers_aberrants = (df['prix_nuit_eur'] <= 0) | (df['nb_nuits'] > 30) | (df['nb_personnes'] <= 0)
print("\n--- RÉPONSE QUESTION 9 : Dossiers anormaux par mois ---")
print(df[dossiers_aberrants].groupby("mois_sejour").size())

# 7. Convertir les colonnes de comptage en entiers propres
df_clean['nb_personnes'] = df_clean['nb_personnes'].astype(int)
df_clean['delai_reservation_j'] = df_clean['delai_reservation_j'].astype(int)
df_clean['nuitees_cumulees_client'] = df_clean['nuitees_cumulees_client'].astype(int)
print(df_clean[['nb_personnes', 'delai_reservation_j', 'nuitees_cumulees_client']].dtypes)

# 8. Sauvegarder le fichier propre et préparer les colonnes numériques pour la suite
df_clean.to_csv('./data/Valmont_cleaned.csv', index=False)
print(f"Nettoyage terminé. Il reste {len(df_clean)} réservations.")

df = df_clean.copy()
df["annule_num"] = (df["a_annule"] == "oui").astype(int)
df["weekend_num"] = (df["weekend"] == "oui").astype(int)
df["pt_dej_num"] = (df["petit_dejeuner"] == "oui").astype(int)

Nombre de réservations : 940
Nombre d'hôtels : 32
Les deux fichiers ont été fusionnés.
Nombre de lignes après la fusion : 940
Nombre de doublons trouvés : 140
Nombre de lignes restantes : 800
Valeurs manquantes avant le remplacement :
nb_personnes      20
petit_dejeuner    20
dtype: int64
nb_personnes      0
petit_dejeuner    0
dtype: int64
Nombre de réservations écartées : 6
Nombre de réservations gardées : 794

--- RÉPONSE QUESTION 9 : Dossiers anormaux par mois ---
mois_sejour
1    1
2    1
3    2
5    1
6    1
dtype: int64
nb_personnes               int64
delai_reservation_j        int64
nuitees_cumulees_client    int64
dtype: object
Nettoyage terminé. Il reste 794 réservations.


In [7]:
# TACHE 2 : ANALYSE EXPLORATOIRE DES ANNULATIONS (Q1 & Q2)
print("Taux d'annulation global :", round(df["annule_num"].mean() * 100, 1), "%")
print((df.groupby("canal")["annule_num"].mean() * 100).round(1))
print((df.groupby("segment_declare")["annule_num"].mean() * 100).round(1))
cols_comp = ["delai_reservation_j", "nb_personnes", "nb_nuits", "prix_nuit_eur"]
print(df.groupby("a_annule")[cols_comp].mean().round(1))
df["tranche_delai"] = pd.cut(df["delai_reservation_j"], bins=[-1, 7, 30, 60, 120, 300],
                             labels=["0-7j", "8-30j", "31-60j", "61-120j", "+120j"])
print((df.groupby("tranche_delai", observed=False)["annule_num"].mean() * 100).round(1))

Taux d'annulation global : 26.1 %
canal
OTA           27.9
agence        22.0
direct        26.7
entreprise    23.7
Name: annule_num, dtype: float64
segment_declare
affaires    16.4
groupe      33.3
loisir      32.2
Name: annule_num, dtype: float64
          delai_reservation_j  nb_personnes  nb_nuits  prix_nuit_eur
a_annule                                                            
non                      36.0           3.1       2.9          198.2
oui                      59.9           4.0       3.0          187.9
tranche_delai
0-7j       15.8
8-30j      19.9
31-60j     27.3
61-120j    32.2
+120j      67.4
Name: annule_num, dtype: float64


In [8]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
# TACHE 3 : SEGMENTATION K-MEANS (Q7 & DEBUT Q8)
cols_seg = ["nb_nuits", "nb_personnes", "delai_reservation_j", "prix_nuit_eur",
            "nuitees_cumulees_client", "weekend_num", "pt_dej_num"]
X_seg = StandardScaler().fit_transform(df[cols_seg])
for k in range(2, 6):
    km_test = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_seg)
    print(f"k={k} -> Inertie : {km_test.inertia_:.1f} | Silhouette : {silhouette_score(X_seg, km_test.labels_):.3f}")
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
df["segment_kmeans"] = kmeans.fit_predict(X_seg)
print(df.groupby("segment_kmeans")[cols_seg + ["annule_num"]].mean().round(2))

SyntaxError: unterminated f-string literal (detected at line 10) (1288100066.py, line 10)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
# TACHE 4 : MODELE DE PRIX ET ROBUSTESSE EN HAUTE SAISON (Q3 & Q4)
df["haute_saison"] = df["mois_sejour"].isin([6, 7, 8, 12]).astype(int)
X_prix = pd.get_dummies(df[["categorie", "weekend_num", "haute_saison", 
"distance_gare_km"]], drop_first=True)
y_prix = df["prix_nuit_eur"]
X_tr, X_te, y_tr, y_te = train_test_split(X_prix, y_prix, test_size=0.3, random_state=42)
reg = LinearRegression().fit(X_tr, y_tr)
y_pred = reg.predict(X_te)
print(f"RMSE test : {mean_squared_error(y_te, y_pred)**0.5:.2f} EUR | R2 : 
{r2_score(y_te, y_pred):.3f}")
test_eval = X_te.copy()
test_eval["erreur_abs"] = (y_te - y_pred).abs()
print(test_eval.groupby("haute_saison")["erreur_abs"].mean().round(2))

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
# TACHE 5 : PREDICTION DES ANNULATIONS (Q5) ET OVERBOOKING (Q6)
cols_knn = ["delai_reservation_j", "nb_personnes", "nb_nuits", "prix_nuit_eur",
            "nuitees_cumulees_client", "weekend_num"]
X_ann, y_ann = df[cols_knn], df["annule_num"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
meilleur_k, meilleur_score = 1, 0
for k in [3, 5, 7, 11, 15, 21, 31]:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    score = cross_val_score(pipe, X_ann, y_ann, cv=cv).mean()
    if score > meilleur_score:
        meilleur_score, meilleur_k = score, k
modele_knn = make_pipeline(StandardScaler(), 
KNeighborsClassifier(n_neighbors=meilleur_k)).fit(X_ann, y_ann)
hotels = df[["hotel_id", "ville", "categorie", "chambres"]].drop_duplicates().copy()
hotels["overbooking_max"] = (hotels["chambres"] * 0.08).astype(int)
print(hotels.head(5))

In [ ]:
# TACHE 6 : CHIFFRAGE DES PERTES ET GAINS SUR CHAMBRES VIDES (FIN Q8)
annulations = df[df["annule_num"] == 1]
nuitees_perdues = annulations["nb_nuits"].sum()
ca_perdu = (annulations["nb_nuits"] * annulations["prix_nuit_eur"]).sum()
gain_estime = ca_perdu * 0.35
print(f"Nuitees perdues : {nuitees_perdues} | Manque a gagner brut : {ca_perdu:,.2f} 
EUR")
print(f"Gain net estime (35% recuperes) : {gain_estime:,.2f} EUR")